## Analysis extension #1b 
### Audited paired MLM retraining: joint and development-only - FINAL

This notebook is the **definitive representation-training pipeline for the revision**. It is intentionally separate from the original `Data_assembly.ipynb`, `Data_assembly2.ipynb`, `Bert.ipynb`, and from `AE1.ipynb`.

The original notebooks are retained unchanged as the historical analysis. AE1 documented what they actually did. AE1b applies only the concrete implementation corrections identified by that audit and then trains a **paired set of models from one common preprocessing pipeline**:

- clean **joint** MLM: development + Burdziak cells contribute to representation learning;
- clean **development-only** MLM: the same developmental corpus is used, but Burdziak cells do not contribute to training.

The two model families use the same gene vocabulary, developmental baseline, tokenization, architecture, hyperparameters, seeds and masking implementation. The intended difference is whether cancer cells contribute to representation learning.

## Corrections made here

1. The developmental training corpus is rebuilt from the serialized **pre-balance** developmental object using the explicit cell-type whitelist from `Data_assembly.ipynb`. This removes the unintended conjunctival-goblet inclusion before balancing.
2. Rare cancer-symbol aliases that map to the same Ensembl ID are handled by **keeping the first occurrence**, matching the behavior of the original helper function. The misleading legacy `duplicate_agg="sum"` interface is removed rather than changing the underlying method.
3. The 10% random-token branch of MLM corruption samples **gene-token IDs only**; `[DEV]` and `[CANCER]` cannot be inserted at gene positions.
4. A single audited preprocessing/tokenization implementation is used for both joint and development-only training.
5. Training is paired by seed (`17`, `42` by default).

## What is *not* changed here

- The revised pancreatic developmental directions are **not trained into the MLM**. AE3 defines those biological populations after embedding. The new models produce new coordinates; AE3 then reconstructs the same finalized biological contrasts inside each representation.

- Likewise, thyroid/thymic/parathyroid epithelial cells are not automatically removed merely because they were previously misused as a pancreatic “foregut” root. They can remain part of a broad developmental representation; AE3 simply does not use them to define pancreatic developmental directions.


## 0. Configuration and output contract

All new outputs live under one versioned root. Downstream AE2-AE5 notebooks read the manifest written here rather than guessing model/data paths.


In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable, Dict, Optional, Tuple
import os, re, json, gzip, time, random, gc, warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import requests

import torch
from torch.utils.data import Dataset
from transformers import (BertConfig,BertForMaskedLM,Trainer,TrainingArguments)

Project paths

In [ ]:
# project_root can be changed for the local path
PROJECT_ROOT = Path("/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC")
DATA_ROOT = PROJECT_ROOT / "data"

# Historical serialized objects used as audited upstream inputs.
DEV_UNBALANCED_OLD = DATA_ROOT / "development/adata_dev_endoderm.h5ad"
CANCER_FULL_OLD = DATA_ROOT / "pdac/burdziak_epifilterout_kept_N1_N2.h5ad"
GTF_PATH = DATA_ROOT / "ref/Mus_musculus.GRCm39.111.gtf"

# Historical aligned matrices are used ONLY as a vocabulary/mapping audit target.
OLD_DEV_ALIGNED = DATA_ROOT / "development/A_dev_aligned.h5ad"
OLD_CANCER_ALIGNED = DATA_ROOT / "pdac/A_cancer_aligned.h5ad"

# Definitive revision outputs.
AUDITED_ROOT = DATA_ROOT / "revision_audited_mlm"
DEV_OUTDIR = AUDITED_ROOT / "development"
CANCER_OUTDIR = AUDITED_ROOT / "pdac"
PREPROC_OUTDIR = AUDITED_ROOT / "preprocessing"
MODEL_ROOT = AUDITED_ROOT / "models"
TRAINER_ROOT = AUDITED_ROOT / "trainer_runs"

for p in [AUDITED_ROOT, DEV_OUTDIR, CANCER_OUTDIR, PREPROC_OUTDIR, MODEL_ROOT, TRAINER_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

DEV_BALANCED_CLEAN = DEV_OUTDIR / "adata_dev_endoderm_balanced_clean.h5ad"
DEV_ALIGNED_CLEAN = DEV_OUTDIR / "A_dev_aligned_clean.h5ad"
CANCER_ALIGNED_CLEAN = CANCER_OUTDIR / "A_cancer_aligned_clean.h5ad"
MANIFEST_PATH = AUDITED_ROOT / "manifest.json"

# ------------------------------------------------------------
# Model/training constants: preserve the original architecture.
# ------------------------------------------------------------
TOP_K = 128
PAD_ID = 0
CLS_ID = 1
SEP_ID = 2
MASK_ID = 3

HIDDEN_SIZE = 256
N_LAYERS = 6
N_HEADS = 8
INTERMEDIATE_SIZE = 512

MLM_PROBABILITY = 0.15
N_EPOCHS = 10
LEARNING_RATE = 5e-4
SEEDS = [17, 42]

# Reuse an already completed clean seed if it exists.
REUSE_EXISTING_MODELS = True

# The original mapping used external rescue for highly detected unresolved symbols.
# AE1b first reuses mappings recoverable from the historical aligned object,
# then optionally calls the same external services for unresolved high-detection genes.
USE_EXTERNAL_RESOLVER = False
EXTERNAL_RESOLVE_MIN_DETECT_FRAC = 0.20
EXTERNAL_ALIAS_CACHE = PREPROC_OUTDIR / "external_alias_cache.json"

print("AUDITED_ROOT:", AUDITED_ROOT)
print("Manifest:", MANIFEST_PATH)


AUDITED_ROOT: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm
Manifest: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/manifest.json


## 1. Rebuild the clean developmental training corpus

The original `Data_assembly.ipynb` defined an explicit set of intended developmental cell types, but the actual broad mapping used substring fallbacks; `"goblet"` therefore also admitted conjunctival goblet cells. Here we start from the serialized pre-balance object and apply the intended exact whitelist **before rebalancing**.

This deliberately preserves the broad developmental representation and the original max-4,000-per-coarse-state balancing strategy. The old coarse labels are retained only as balancing strata/provenance; AE3 uses original Qiu `celltype_update` plus developmental time for biological directions.


In [ ]:
INTENDED_CELLTYPES = {
    # Pancreas / hepatobiliary / foregut core
    "Pancreatic acinar cells",
    "Foregut epithelial cells",
    "Biliary epithelial cells",
    "Hepatocytes",

    # Endocrine reference
    "Pancreatic islets",

    # Gut epithelium
    "Gut",
    "Midgut/Hindgut epithelial cells",
    "Intestinal goblet cells",
    "Intestinal enteroendocrine cells",

    # Lung & airway
    "Lung progenitor cells",
    "Lung cells (Eln+)",
    "Airway goblet cells",
    "Airway club cells",
    "Alveolar Type 1 cells",
    "Alveolar Type 2 cells",

    # Other foregut-associated epithelial organs retained in the broad encoder
    "Thyroid gland cells",
    "Parathyroid epithelial cells",
    "Thymic epithelial cells",
}

EXPECTED_AUDIT_EXCLUDED_TYPES = {"Conjunctival goblet cells"}

A_dev_unbalanced = sc.read_h5ad(DEV_UNBALANCED_OLD)
if "celltype_update" not in A_dev_unbalanced.obs:
    raise KeyError("Expected A_dev_unbalanced.obs['celltype_update'].")

ct = A_dev_unbalanced.obs["celltype_update"].astype(str)
keep_exact = ct.isin(INTENDED_CELLTYPES)

excluded_counts = ct[~keep_exact].value_counts()
print("Cells excluded by the exact intended whitelist:")
display(excluded_counts.to_frame("n_cells"))

unexpected_excluded = set(excluded_counts.index) - EXPECTED_AUDIT_EXCLUDED_TYPES
if unexpected_excluded:
    raise RuntimeError(
        "The exact whitelist would remove additional cell types not identified by AE1. "
        "Inspect before continuing: " + repr(sorted(unexpected_excluded))
    )

if "Conjunctival goblet cells" in set(ct[keep_exact]):
    raise RuntimeError("Conjunctival goblet cells remain after the clean whitelist.")

A_dev_clean_unbalanced = A_dev_unbalanced[keep_exact.to_numpy()].copy()

# Preserve the original coarse balancing strata as legacy/provenance only.
# ------------------------------------------------------------------------
if "dev_state_union" not in A_dev_clean_unbalanced.obs:
    raise KeyError("Expected legacy balancing stratum 'dev_state_union'.")

def downsample_by(obs, key, max_per=4000, seed=13):
    selected = []
    for cat, sub in obs.groupby(key, observed=True):
        n = min(len(sub), max_per)
        selected.append(sub.sample(n=n, random_state=seed).index.to_numpy())
    return np.concatenate(selected)

BALANCE_SEED = 13
MAX_PER_STATE = 4000

keep_idx = downsample_by(
    A_dev_clean_unbalanced.obs,
    key="dev_state_union",
    max_per=MAX_PER_STATE,
    seed=BALANCE_SEED,
)
A_dev_bal = A_dev_clean_unbalanced[keep_idx].copy()

print("\nClean balanced developmental corpus:")
display(A_dev_bal.obs["dev_state_union"].astype(str).value_counts().to_frame("n_cells"))
print("Total:", A_dev_bal.n_obs)

if (A_dev_bal.obs["celltype_update"].astype(str) == "Conjunctival goblet cells").any():
    raise RuntimeError("Clean balanced corpus still contains conjunctival goblet cells.")

A_dev_bal.write_h5ad(DEV_BALANCED_CLEAN, compression="lzf")
print("Saved:", DEV_BALANCED_CLEAN)


Cells excluded by the exact intended whitelist:


,n_cells
celltype_update,
Conjunctival goblet cells,1244



Clean balanced developmental corpus:


,n_cells
dev_state_union,
gut_epithelium_distal,4000
ductal,4000
lung_progenitor,4000
lung_airway,4000
gut_epithelium_proximal_or_mixed,4000
liver_hepatocyte,4000
foregut_other,3948
acinar_or_tip,3747
biliary,1742


Total: 34760
Saved: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/development/adata_dev_endoderm_balanced_clean.h5ad


### 1A. Load the unchanged Burdziak cohort as raw counts

AE1 found no analogous cell-selection bug in the progression cohort. The complete 28,131-cell cohort is therefore retained. As in the original alignment notebook, raw counts are taken from `.raw`.


In [ ]:
A_cancer_original = sc.read_h5ad(CANCER_FULL_OLD)

if A_cancer_original.raw is None:
    raise RuntimeError("The audited Burdziak object has no .raw counts.")

A_cancer_raw = ad.AnnData(
    X=A_cancer_original.raw.X.copy(),
    obs=A_cancer_original.obs.copy(),
    var=A_cancer_original.raw.var.copy(),
)

print("Burdziak raw matrix:", A_cancer_raw.shape)
if "condition" in A_cancer_raw.obs:
    display(A_cancer_raw.obs["condition"].astype(str).value_counts().sort_index().to_frame("n_cells"))
if "cancer_state" in A_cancer_raw.obs:
    display(A_cancer_raw.obs["cancer_state"].astype(str).value_counts().to_frame("n_cells"))


Burdziak raw matrix: (28131, 16826)


,n_cells
condition,
K1,4503
K1.5,2476
K2,2079
K3,3338
K4,1254
K5,5364
K6,3491
N1,3225
N2,2401


,n_cells
cancer_state,
transitional,13650
malignant_ductal,8855
ductal,5626


## 2. Audited gene harmonization

This follows `Data_assembly2.ipynb` while making the duplicate-handling semantics explicit.

Important safeguards:

- Ensembl version suffixes are stripped from development.
- Cancer symbols are mapped through the same GRCm39/Ensembl 111 GTF.
- Previously recovered symbol→Ensembl mappings are reused from the historical aligned matrix when available, making the revised mapping less dependent on live external services.
- Remaining highly detected unresolved symbols can be queried using the same Ensembl/MyGene strategy as the original notebook and cached.
- If multiple cancer symbols map to the same Ensembl ID, the revised pipeline **keeps the first occurrence and drops subsequent aliases**, matching the behavior of the original helper function. The old `duplicate_agg="sum"` argument was misleading API/documentation and is not used here.
- Before training, the final common vocabulary is compared with the historical 15,376-gene vocabulary. A change in vocabulary stops the notebook rather than silently altering the experiment.


In [5]:
def strip_ens_version(ids: Iterable[str]) -> pd.Index:
    return pd.Index([str(g).split(".")[0] for g in ids])

def parse_gtf_symbol_maps(gtf_path: Path):
    ens2sym = {}
    sym2ens_upper = {}

    def parse_attrs(s):
        out = {}
        for field in s.strip().split(";"):
            field = field.strip()
            if not field or " " not in field:
                continue
            k, v = field.split(" ", 1)
            out[k] = v.strip().strip('"')
        return out

    opener = gzip.open if str(gtf_path).endswith(".gz") else open
    with opener(gtf_path, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9 or parts[2] != "gene":
                continue
            attrs = parse_attrs(parts[8])
            gid = attrs.get("gene_id")
            gname = attrs.get("gene_name")
            if not gid or not gname:
                continue
            gid = gid.split(".")[0]
            ens2sym.setdefault(gid, gname)
            sym2ens_upper.setdefault(gname.upper(), gid)
    return ens2sym, sym2ens_upper

ens2sym, sym2ens_upper = parse_gtf_symbol_maps(GTF_PATH)
print("GTF genes:", len(ens2sym))


def collapse_duplicate_var_names_keepfirst(adata):
    """
    Collapse duplicate var_names by keeping the first occurrence.

    This matches the actual behavior of the original helper function.
    The old outer API exposed a `duplicate_agg="sum"` option, but that
    argument was not used by the implementation. We therefore preserve
    the original analytical behavior and remove the misleading aggregation
    semantics rather than introducing a new summation rule.
    """
    names = pd.Index(adata.var_names.astype(str))

    counts = pd.Series(names).value_counts()
    dup_table = (
        counts[counts > 1]
        .rename_axis("ensembl_id")
        .rename("n_input_columns")
        .reset_index()
    )

    if not names.has_duplicates:
        return adata.copy(), dup_table

    keep = ~names.duplicated(keep="first")
    out = adata[:, keep].copy()

    return out, dup_table


# Historical common-gene mapping is useful as a deterministic alias cache.
legacy_alias = {}
legacy_common = None
if OLD_CANCER_ALIGNED.exists():
    A_old_can = sc.read_h5ad(OLD_CANCER_ALIGNED)
    legacy_common = A_old_can.var_names.astype(str).to_numpy()
    if "symbol_orig" in A_old_can.var:
        for gid, sym in zip(A_old_can.var_names.astype(str), A_old_can.var["symbol_orig"].astype(str)):
            legacy_alias.setdefault(sym, gid)
    print("Recovered legacy alias mappings:", len(legacy_alias))

# Existing external cache, if any.
external_alias = {}
if EXTERNAL_ALIAS_CACHE.exists():
    with open(EXTERNAL_ALIAS_CACHE) as f:
        external_alias = {str(k): str(v) for k, v in json.load(f).items()}
    print("Loaded external alias cache:", len(external_alias))


GTF genes: 56655
Recovered legacy alias mappings: 15376
Loaded external alias cache: 12


In [6]:
ENSEMBL_SERVER = "https://rest.ensembl.org"
MYGENE_SERVER = "https://mygene.info/v3"

def query_ensembl_symbol_mouse(symbol, max_retries=3, sleep_sec=0.1):
    url = ENSEMBL_SERVER + f"/xrefs/symbol/mus_musculus/{symbol}?"
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers={"Content-Type": "application/json"}, timeout=10)
            if r.status_code == 404:
                return None
            if not r.ok:
                time.sleep(sleep_sec)
                continue
            data = r.json()
            if not data:
                return None
            genes = [x for x in data if x.get("type") == "gene"] or data
            return genes[0].get("id")
        except Exception:
            if attempt == max_retries - 1:
                return None
            time.sleep(sleep_sec)
    return None

def query_mygene_symbol_mouse(symbol, max_retries=3, sleep_sec=0.1):
    params = {
        "q": f"symbol:{symbol}",
        "species": "mouse",
        "fields": "ensembl.gene,symbol",
        "size": 5,
    }
    for attempt in range(max_retries):
        try:
            r = requests.get(MYGENE_SERVER + "/query", params=params, timeout=10)
            if not r.ok:
                time.sleep(sleep_sec)
                continue
            hits = r.json().get("hits", [])
            for h in sorted(hits, key=lambda z: z.get("_score", 0), reverse=True):
                ens = h.get("ensembl")
                if isinstance(ens, list) and ens:
                    return ens[0].get("gene")
                if isinstance(ens, dict):
                    return ens.get("gene")
            return None
        except Exception:
            if attempt == max_retries - 1:
                return None
            time.sleep(sleep_sec)
    return None

def map_cancer_symbols(symbols, X):
    symbols = pd.Index(symbols.astype(str))
    mapped = pd.Series(index=symbols, dtype=object)

    # 1) GTF
    for s in symbols:
        gid = sym2ens_upper.get(s.upper())
        if gid is not None:
            mapped.loc[s] = gid

    # 2) Frozen mappings recoverable from the original aligned result.
    for s in symbols[mapped.isna()]:
        if s in legacy_alias:
            mapped.loc[s] = legacy_alias[s]

    # 3) Previously cached external mappings.
    for s in symbols[mapped.isna()]:
        if s in external_alias:
            mapped.loc[s] = external_alias[s]

    unresolved = symbols[mapped.isna()]
    print("Unresolved before optional external rescue:", len(unresolved))

    if USE_EXTERNAL_RESOLVER and len(unresolved):
        # Restrict live queries exactly as the original pipeline did: only
        # high-detection unresolved genes.
        col_mask = symbols.isin(unresolved)
        Xu = X[:, col_mask]
        if sp.issparse(Xu):
            detect = np.asarray((Xu > 0).mean(axis=0)).ravel()
        else:
            detect = (np.asarray(Xu) > 0).mean(axis=0)

        unresolved_df = pd.DataFrame({
            "symbol": symbols[col_mask],
            "detect_frac": detect,
        })
        to_query = unresolved_df.loc[
            unresolved_df["detect_frac"] >= EXTERNAL_RESOLVE_MIN_DETECT_FRAC,
            "symbol"
        ].astype(str).tolist()

        print("External queries requested:", len(to_query))
        for i, s in enumerate(to_query):
            gid = query_ensembl_symbol_mouse(s)
            if gid is None:
                gid = query_mygene_symbol_mouse(s)
            if gid:
                gid = str(gid).split(".")[0]
                external_alias[s] = gid
                mapped.loc[s] = gid
            if (i + 1) % 20 == 0:
                print(f"  external mapping {i+1}/{len(to_query)}")
            time.sleep(0.1)

        with open(EXTERNAL_ALIAS_CACHE, "w") as f:
            json.dump(external_alias, f, indent=2, sort_keys=True)

    return mapped


# ---- Development: strip versions and keep first any accidental duplicate IDs.
A_dev_map = A_dev_bal.copy()
A_dev_map.var_names = strip_ens_version(A_dev_map.var_names)
A_dev_map, dev_dup = collapse_duplicate_var_names_keepfirst(A_dev_map)
A_dev_map.var["symbol"] = [ens2sym.get(g, g) for g in A_dev_map.var_names]

# ---- Cancer: symbol -> Ensembl.
orig_symbols = A_cancer_raw.var_names.astype(str)
mapped = map_cancer_symbols(orig_symbols, A_cancer_raw.X)
mapped_mask = mapped.notna().to_numpy()

A_can_map = A_cancer_raw[:, mapped_mask].copy()
kept_symbols = orig_symbols[mapped_mask]
kept_ids = mapped.iloc[np.flatnonzero(mapped_mask)].astype(str).to_numpy()

A_can_map.var_names = pd.Index(kept_ids)
A_can_map.var["symbol_orig"] = kept_symbols
A_can_map.var["symbol"] = [ens2sym.get(g, g) for g in A_can_map.var_names]

A_can_map, cancer_dup = collapse_duplicate_var_names_keepfirst(A_can_map)

print("\nDuplicate Ensembl mappings handled by KEEP-FIRST (original helper behavior):")
display(cancer_dup)
print("n duplicated Ensembl IDs:", len(cancer_dup))

common = pd.Index(A_dev_map.var_names).intersection(A_can_map.var_names).sort_values()

# Strong audit guard: the correction should change counts, not silently change
# the shared vocabulary. If the historical aligned object exists, require exact
# common-gene identity/order before training.
if legacy_common is not None:
    legacy_common = pd.Index(np.asarray(legacy_common, dtype=str))
    available_now = (set(A_dev_map.var_names)& set(A_can_map.var_names))

    missing_legacy = [
        g for g in legacy_common
        if g not in available_now
    ]

    extra_now = sorted(available_now - set(legacy_common))

    if missing_legacy:
        raise RuntimeError(
            "Historical training genes can no longer be reconstructed.\n"
            f"Missing n={len(missing_legacy)}\n"
            f"Examples={missing_legacy[:20]}"
        )

    print("PASS: all historical training genes are available.")
    print(f"Additional currently resolvable genes excluded: "f"{len(extra_now)}")
    print("Examples:", extra_now[:20])

    # Freeze the definitive retraining vocabulary
    # to the exact historical model vocabulary/order.
    common = legacy_common.copy()

A_dev_aln = A_dev_map[:, common].copy()
A_can_aln = A_can_map[:, common].copy()

A_dev_aln.var["symbol"] = [ens2sym.get(g, g) for g in common]
A_can_aln.var["symbol"] = A_dev_aln.var["symbol"].to_numpy()

A_dev_aln.write_h5ad(DEV_ALIGNED_CLEAN, compression="lzf")
A_can_aln.write_h5ad(CANCER_ALIGNED_CLEAN, compression="lzf")

alignment_summary = {
    "n_dev_cells": int(A_dev_aln.n_obs),
    "n_cancer_cells": int(A_can_aln.n_obs),
    "n_common_genes": int(len(common)),
    "n_cancer_symbols_input": int(len(orig_symbols)),
    "n_cancer_symbols_mapped_before_collapse": int(mapped_mask.sum()),
    "n_duplicate_ensembl_alias_groups_keepfirst": int(len(cancer_dup)),
    "n_cancer_genes_after_duplicate_keepfirst": int(A_can_map.n_vars),
    "vocabulary_matches_historical_exactly": bool(
        legacy_common is not None and np.array_equal(common.to_numpy(dtype=str), legacy_common)
    ),
}
with open(PREPROC_OUTDIR / "alignment_summary.json", "w") as f:
    json.dump(alignment_summary, f, indent=2)

cancer_dup.to_csv(PREPROC_OUTDIR / "duplicate_ensembl_aliases_keepfirst.csv", index=False)

print("Saved clean aligned development:", DEV_ALIGNED_CLEAN)
print("Saved clean aligned cancer:     ", CANCER_ALIGNED_CLEAN)
print(alignment_summary)


Unresolved before optional external rescue: 876

Duplicate Ensembl mappings handled by KEEP-FIRST (original helper behavior):


,ensembl_id,n_input_columns
0,ENSMUSG00000004891,2
1,ENSMUSG00000072694,2
2,ENSMUSG00000027803,2


n duplicated Ensembl IDs: 3
PASS: all historical training genes are available.
Additional currently resolvable genes excluded: 2
Examples: ['ENSMUSG00000022956', 'ENSMUSG00000025144']
Saved clean aligned development: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/development/A_dev_aligned_clean.h5ad
Saved clean aligned cancer:      /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/pdac/A_cancer_aligned_clean.h5ad
{'n_dev_cells': 34760, 'n_cancer_cells': 28131, 'n_common_genes': 15376, 'n_cancer_symbols_input': 16826, 'n_cancer_symbols_mapped_before_collapse': 15950, 'n_duplicate_ensembl_alias_groups_keepfirst': 3, 'n_cancer_genes_after_duplicate_keepfirst': 15947, 'vocabulary_matches_historical_exactly': True}


further check

In [7]:
A_can_old = sc.read_h5ad(OLD_CANCER_ALIGNED)

print("Old:", A_can_old.shape)
print("New:", A_can_aln.shape)

print(
    "Same cells:",
    np.array_equal(
        A_can_old.obs_names.astype(str),
        A_can_aln.obs_names.astype(str)
    )
)

print(
    "Same genes/order:",
    np.array_equal(
        A_can_old.var_names.astype(str),
        A_can_aln.var_names.astype(str)
    )
)

X_old = A_can_old.X
X_new = A_can_aln.X

if sp.issparse(X_old):
    X_old = X_old.tocsr()
if sp.issparse(X_new):
    X_new = X_new.tocsr()

if sp.issparse(X_old) and sp.issparse(X_new):
    D = X_new - X_old
    print("Different entries:", D.nnz)
    print(
        "Maximum absolute difference:",
        0.0 if D.nnz == 0 else np.max(np.abs(D.data))
    )
else:
    xo = X_old.toarray() if sp.issparse(X_old) else np.asarray(X_old)
    xn = X_new.toarray() if sp.issparse(X_new) else np.asarray(X_new)
    print("Different entries:", np.count_nonzero(xn != xo))
    print("Maximum absolute difference:", np.max(np.abs(xn - xo)))

Old: (28131, 15376)
New: (28131, 15376)
Same cells: True
Same genes/order: True
Different entries: 0
Maximum absolute difference: 0.0


>### TOKENIZER SECTION

## 3. One shared audited tokenizer for both model families

The developmental baseline is fitted **once** from the clean aligned developmental corpus and is shared by joint and development-only training.

The sequence remains:

`[DOMAIN] [CLS] top-128 developmental-relative gene tokens [SEP]`

For the joint model:
- developmental cells use `[DEV]`;
- Burdziak cells use `[CANCER]`.

For the development-only model:
- only developmental `[DEV]` sequences enter training;
- after training, both developmental and Burdziak query cells are encoded with the learned `[DEV]` token because `[CANCER]` has never been trained.

The dataset label stored in `domain.npy` remains `0=development, 1=Burdziak`; it records query provenance, not necessarily the literal first token used by the development-only encoder.


In [ ]:
# Vocabulary
genes = A_dev_aln.var_names.astype(str).to_numpy()
assert np.array_equal(genes, A_can_aln.var_names.astype(str).to_numpy())

gene2id = {g: i + 4 for i, g in enumerate(genes)}
N_GENES = len(genes)

DEV_DOM_ID = 4 + N_GENES
CANCER_DOM_ID = DEV_DOM_ID + 1
VOCAB_SIZE = 4 + N_GENES + 2

def library_normalized_matrix(X, target_sum=1e4):
    if sp.issparse(X):
        X = X.tocsr().astype(np.float32)
        lib = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
        safe = np.where(lib > 0, lib, 1.0).astype(np.float32)
        scale = (np.float32(target_sum) / safe).astype(np.float32)
        return X.multiply(scale[:, None]).tocsr(), lib
    X = np.asarray(X, dtype=np.float32)
    lib = X.sum(axis=1).astype(np.float32)
    safe = np.where(lib > 0, lib, 1.0).astype(np.float32)
    return (X * (np.float32(target_sum) / safe)[:, None]).astype(np.float32), lib

def nonzero_median_per_gene(X_norm):
    n_genes = X_norm.shape[1]
    med = np.zeros(n_genes, dtype=np.float32)
    if sp.issparse(X_norm):
        Xc = X_norm.tocsc()
        for j in range(n_genes):
            vals = Xc.data[Xc.indptr[j]:Xc.indptr[j+1]]
            med[j] = np.median(vals) if vals.size else 0.0
    else:
        Xd = np.asarray(X_norm, dtype=np.float32)
        for j in range(n_genes):
            vals = Xd[:, j]
            vals = vals[vals > 0]
            med[j] = np.median(vals) if vals.size else 0.0
    return med

X_dev_norm, dev_library_sizes = library_normalized_matrix(A_dev_aln.X)
dev_baseline = nonzero_median_per_gene(X_dev_norm)
baseline_zero = dev_baseline <= 0

np.save(PREPROC_OUTDIR / "dev_baseline.npy", dev_baseline)
np.save(PREPROC_OUTDIR / "dev_library_sizes.npy", dev_library_sizes)
np.save(PREPROC_OUTDIR / "gene_vocabulary.npy", genes)

print("Genes:", N_GENES)
print("Zero developmental baseline:", int(baseline_zero.sum()))
print("DEV_DOM_ID:", DEV_DOM_ID, "CANCER_DOM_ID:", CANCER_DOM_ID, "VOCAB_SIZE:", VOCAB_SIZE)


def cell_to_base_tokens(X_row, baseline, top_k=TOP_K, target_sum=1e4):
    """
    Audited definitive tokenizer.
    The ranking semantics intentionally remain the same as the original/AE4
    implementation; the clean paired models share this exact function.
    """
    if sp.issparse(X_row):
        raw = X_row.toarray().ravel().astype(np.float32)
    else:
        raw = np.asarray(X_row).ravel().astype(np.float32)

    lib = float(raw.sum())
    if lib <= 0:
        return np.asarray([CLS_ID, SEP_ID] + [PAD_ID] * top_k, dtype=np.int64)

    normed = raw / lib * float(target_sum)

    safe = baseline.copy()
    z = safe <= 0
    safe[z] = 1.0

    rel = normed / safe
    rel[z] = 0.0

    scores = np.log1p(np.maximum(rel, 0.0) + 1e-6)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

    if np.all(scores == 0):
        return np.asarray([CLS_ID, SEP_ID] + [PAD_ID] * top_k, dtype=np.int64)

    k = min(top_k, scores.size)
    idx_k = np.argpartition(-scores, k - 1)[:k]
    idx_sorted = idx_k[np.argsort(-scores[idx_k], kind="stable")]

    ids = [CLS_ID] + (idx_sorted[:top_k] + 4).tolist() + [SEP_ID]
    if len(ids) < top_k + 2:
        ids += [PAD_ID] * (top_k + 2 - len(ids))
    return np.asarray(ids, dtype=np.int64)

def matrix_to_base_tokens(A, baseline):
    out = np.full((A.n_obs, TOP_K + 2), PAD_ID, dtype=np.int64)
    for i in range(A.n_obs):
        out[i] = cell_to_base_tokens(A.X[i], baseline)
        if (i + 1) % 5000 == 0 or i + 1 == A.n_obs:
            print(f"tokenized {i+1:,}/{A.n_obs:,}")
    return out

def add_domain_token(base_tokens, domain_id):
    N, L = base_tokens.shape
    out = np.full((N, L + 1), PAD_ID, dtype=np.int64)
    out[:, 0] = int(domain_id)
    out[:, 1:] = base_tokens
    attn = (out != PAD_ID).astype(np.int64)
    return out, attn

dev_base_tokens = matrix_to_base_tokens(A_dev_aln, dev_baseline)
can_base_tokens = matrix_to_base_tokens(A_can_aln, dev_baseline)

dev_tokens_DEV, dev_attn_DEV = add_domain_token(dev_base_tokens, DEV_DOM_ID)
can_tokens_CANCER, can_attn_CANCER = add_domain_token(can_base_tokens, CANCER_DOM_ID)
can_tokens_DEV, can_attn_DEV = add_domain_token(can_base_tokens, DEV_DOM_ID)

np.save(PREPROC_OUTDIR / "dev_base_tokens.npy", dev_base_tokens)
np.save(PREPROC_OUTDIR / "cancer_base_tokens.npy", can_base_tokens)

print("DEV query tokens:", dev_tokens_DEV.shape)
print("Cancer joint-query tokens:", can_tokens_CANCER.shape)
print("Cancer dev-only-query tokens:", can_tokens_DEV.shape)


Genes: 15376
Zero developmental baseline: 78
DEV_DOM_ID: 15380 CANCER_DOM_ID: 15381 VOCAB_SIZE: 15382
tokenized 5,000/34,760
tokenized 10,000/34,760
tokenized 15,000/34,760
tokenized 20,000/34,760
tokenized 25,000/34,760
tokenized 30,000/34,760
tokenized 34,760/34,760
tokenized 5,000/28,131
tokenized 10,000/28,131
tokenized 15,000/28,131
tokenized 20,000/28,131
tokenized 25,000/28,131
tokenized 28,131/28,131
DEV query tokens: (34760, 131)
Cancer joint-query tokens: (28131, 131)
Cancer dev-only-query tokens: (28131, 131)


## 4. Audited MLM masking and paired training

The only masking correction is in the random-replacement branch:

```python
torch.randint(low=4, high=DEV_DOM_ID, ...)
```

Because `DEV_DOM_ID == 4 + n_genes`, this samples exactly the gene-token interval and excludes both domain tokens.

The same collator is used for joint and development-only training.


In [9]:
class TokenDataset(Dataset):
    def __init__(self, tokens, attn):
        self.tokens = torch.as_tensor(tokens, dtype=torch.long)
        self.attn = torch.as_tensor(attn, dtype=torch.long)

    def __len__(self):
        return self.tokens.shape[0]

    def __getitem__(self, i):
        return {
            "input_ids": self.tokens[i],
            "attention_mask": self.attn[i],
        }

SPECIAL_IDS = {PAD_ID, CLS_ID, SEP_ID, MASK_ID, DEV_DOM_ID, CANCER_DOM_ID}

@dataclass
class AuditedMLMCollator:
    mlm_probability: float = MLM_PROBABILITY

    def __call__(self, batch):
        input_ids = torch.stack([b["input_ids"] for b in batch], dim=0)
        attention_mask = torch.stack([b["attention_mask"] for b in batch], dim=0)
        labels = input_ids.clone()

        maskable = torch.ones_like(input_ids, dtype=torch.bool)
        for sid in SPECIAL_IDS:
            maskable &= input_ids != sid

        prob = torch.full(input_ids.shape, self.mlm_probability, dtype=torch.float32)
        sampled = torch.bernoulli(prob).bool() & maskable

        replace_mask = sampled & (
            torch.rand_like(input_ids, dtype=torch.float32) < 0.8
        )
        random_mask = sampled & (~replace_mask) & (
            torch.rand_like(input_ids, dtype=torch.float32) < 0.5
        )

        input_ids[replace_mask] = MASK_ID

        # AUDIT FIX: gene tokens only: 4 ... DEV_DOM_ID-1.
        rand_ids = torch.randint(
            low=4,
            high=DEV_DOM_ID,
            size=input_ids.shape,
            dtype=torch.long,
        )
        input_ids[random_mask] = rand_ids[random_mask]

        labels[~sampled] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_model():
    seq_len = TOP_K + 3
    config = BertConfig(
        vocab_size=VOCAB_SIZE,
        hidden_size=HIDDEN_SIZE,
        num_hidden_layers=N_LAYERS,
        num_attention_heads=N_HEADS,
        intermediate_size=INTERMEDIATE_SIZE,
        max_position_embeddings=seq_len + 2,
        pad_token_id=PAD_ID,
    )
    return BertForMaskedLM(config)

# Joint and development-only training corpora from the SAME tokenization.
joint_train_tokens = np.vstack([dev_tokens_DEV, can_tokens_CANCER])
joint_train_attn = np.vstack([dev_attn_DEV, can_attn_CANCER])

devonly_train_tokens = dev_tokens_DEV
devonly_train_attn = dev_attn_DEV

print("Joint training cells:", len(joint_train_tokens))
print("Development-only training cells:", len(devonly_train_tokens))


Joint training cells: 62891
Development-only training cells: 34760


In [10]:
@torch.no_grad()
def cls_embeddings(model, tokens, attn, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    chunks = []
    for i in range(0, len(tokens), batch_size):
        x = torch.as_tensor(tokens[i:i+batch_size], dtype=torch.long, device=device)
        a = torch.as_tensor(attn[i:i+batch_size], dtype=torch.long, device=device)
        out = model.bert(input_ids=x, attention_mask=a, return_dict=True)
        # [DOMAIN] [CLS] ... -> CLS is position 1.
        chunks.append(out.last_hidden_state[:, 1, :].cpu())
    return torch.cat(chunks, dim=0).numpy()

def model_dir_for(scope, seed):
    return MODEL_ROOT / scope / f"seed_{seed}" / "model"

def seed_dir_for(scope, seed):
    return MODEL_ROOT / scope / f"seed_{seed}"

def build_eval_corpus(scope):
    if scope == "joint":
        can_tok, can_attn = can_tokens_CANCER, can_attn_CANCER
        query_rule = "DEV_for_development__CANCER_for_burdziak"
    elif scope == "development_only":
        can_tok, can_attn = can_tokens_DEV, can_attn_DEV
        query_rule = "DEV_for_all_queries"
    else:
        raise ValueError(scope)

    Xtok = np.vstack([dev_tokens_DEV, can_tok])
    Xattn = np.vstack([dev_attn_DEV, can_attn])
    biological_domain = np.concatenate([
        np.zeros(A_dev_aln.n_obs, dtype=np.int8),
        np.ones(A_can_aln.n_obs, dtype=np.int8),
    ])
    return Xtok, Xattn, biological_domain, query_rule

def train_one(scope, seed):
    if scope not in {"joint", "development_only"}:
        raise ValueError(scope)

    seed_dir = seed_dir_for(scope, seed)
    model_dir = model_dir_for(scope, seed)
    trainer_dir = TRAINER_ROOT / scope / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    trainer_dir.mkdir(parents=True, exist_ok=True)

    expected_files = [
        model_dir / "config.json",
        model_dir / "vocab_meta.json",
        model_dir / "X_tokens.npy",
        model_dir / "X_attn.npy",
        model_dir / "domain.npy",
        model_dir / "dev_baseline.npy",
    ]
    if REUSE_EXISTING_MODELS and all(p.exists() for p in expected_files):
        print(f"[reuse] {scope} seed {seed}: {model_dir}")
        return model_dir

    set_all_seeds(seed)
    model = make_model()

    if scope == "joint":
        train_tokens = joint_train_tokens
        train_attn = joint_train_attn
    else:
        train_tokens = devonly_train_tokens
        train_attn = devonly_train_attn

    ds = TokenDataset(train_tokens, train_attn)

    args = TrainingArguments(
        output_dir=str(trainer_dir),
        overwrite_output_dir=True,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=N_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        max_grad_norm=1.0,
        fp16=torch.cuda.is_available(),
        dataloader_drop_last=True,
        logging_steps=50,
        save_total_limit=2,
        report_to="none",
        seed=int(seed),
        data_seed=int(seed),
    )

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds,
        data_collator=AuditedMLMCollator(),
    )
    trainer.train()
    model_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(model_dir))

    Xtok_eval, Xattn_eval, domain_eval, query_rule = build_eval_corpus(scope)

    meta = {
        "pipeline_version": "audited_mlm_v1",
        "gene2id": gene2id,
        "PAD_ID": PAD_ID,
        "CLS_ID": CLS_ID,
        "SEP_ID": SEP_ID,
        "MASK_ID": MASK_ID,
        "DEV_DOM_ID": DEV_DOM_ID,
        "CANCER_DOM_ID": CANCER_DOM_ID,
        "vocab_size": VOCAB_SIZE,
        "TOP_K": TOP_K,
        "seed": int(seed),
        "training_scope": scope,
        "query_domain_rule": query_rule,
        "random_replacement_range": [4, int(DEV_DOM_ID - 1)],
        "random_replacement_gene_only": True,
        "developmental_corpus": str(DEV_ALIGNED_CLEAN),
        "cancer_query_corpus": str(CANCER_ALIGNED_CLEAN),
    }

    with open(model_dir / "vocab_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    np.save(model_dir / "X_tokens.npy", Xtok_eval)
    np.save(model_dir / "X_attn.npy", Xattn_eval)
    np.save(model_dir / "domain.npy", domain_eval)
    np.save(model_dir / "dev_obs_names.npy", A_dev_aln.obs_names.astype(str).to_numpy())
    np.save(model_dir / "cancer_obs_names.npy", A_can_aln.obs_names.astype(str).to_numpy())
    np.save(model_dir / "dev_baseline.npy", dev_baseline)

    # Cache definitive CLS embeddings for downstream notebooks.
    Xdev = cls_embeddings(model, Xtok_eval[domain_eval == 0], Xattn_eval[domain_eval == 0])
    Xcan = cls_embeddings(model, Xtok_eval[domain_eval == 1], Xattn_eval[domain_eval == 1])
    np.save(model_dir / "X_dev_cls.npy", Xdev)
    np.save(model_dir / "X_cancer_cls.npy", Xcan)

    pd.DataFrame(trainer.state.log_history).to_csv(
        seed_dir / "training_log.csv", index=False
    )

    del trainer, model, Xdev, Xcan
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[trained] {scope} seed {seed}: {model_dir}")
    return model_dir


## 5. Train/reuse the paired model set

Run both scopes for each seed before proceeding. If a completed audited model directory is present, it is reused rather than overwritten.


In [11]:
model_paths = {"joint": {}, "development_only": {}}

for seed in SEEDS:
    for scope in ["joint", "development_only"]:
        model_paths[scope][str(seed)] = str(train_one(scope, seed))

print(json.dumps(model_paths, indent=2))


[reuse] joint seed 17: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_17/model
[reuse] development_only seed 17: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/development_only/seed_17/model
[reuse] joint seed 42: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_42/model
[reuse] development_only seed 42: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/development_only/seed_42/model
{
  "joint": {
    "17": "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_17/model",
    "42": "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_42/model"
  },
  "development_only": {
    "17": "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/development_only/seed_17/model",
    "42": "/home/jfpoyatos/Wo

## 6. Paired post-training integrity checks

These checks establish the downstream contract:

- all four encoders use the same gene vocabulary and developmental baseline;
- saved token corpora have the same biological cell order;
- joint and development-only models differ in training scope/query-domain rule, not in preprocessing;
- cached embeddings exist for AE2–AE5.


In [12]:
integrity_rows = []

reference_genes = None
reference_baseline = None

for scope in ["joint", "development_only"]:
    for seed in SEEDS:
        model_dir = Path(model_paths[scope][str(seed)])

        with open(model_dir / "vocab_meta.json") as f:
            meta = json.load(f)

        genes_here = np.array(
            [g for g, _ in sorted(meta["gene2id"].items(), key=lambda kv: kv[1])],
            dtype=str,
        )
        baseline_here = np.load(model_dir / "dev_baseline.npy")

        if reference_genes is None:
            reference_genes = genes_here
            reference_baseline = baseline_here
        else:
            assert np.array_equal(genes_here, reference_genes)
            assert np.array_equal(baseline_here, reference_baseline)

        dom = np.load(model_dir / "domain.npy")
        assert int((dom == 0).sum()) == A_dev_aln.n_obs
        assert int((dom == 1).sum()) == A_can_aln.n_obs

        dev_names = np.load(model_dir / "dev_obs_names.npy", allow_pickle=True).astype(str)
        can_names = np.load(model_dir / "cancer_obs_names.npy", allow_pickle=True).astype(str)

        assert np.array_equal(dev_names, A_dev_aln.obs_names.astype(str).to_numpy())
        assert np.array_equal(can_names, A_can_aln.obs_names.astype(str).to_numpy())

        Xdev = np.load(model_dir / "X_dev_cls.npy", mmap_mode="r")
        Xcan = np.load(model_dir / "X_cancer_cls.npy", mmap_mode="r")

        integrity_rows.append({
            "scope": scope,
            "seed": seed,
            "n_genes": len(genes_here),
            "n_dev": Xdev.shape[0],
            "n_cancer": Xcan.shape[0],
            "hidden_size": Xdev.shape[1],
            "query_domain_rule": meta["query_domain_rule"],
            "gene_only_random_replacement": meta["random_replacement_gene_only"],
        })

integrity = pd.DataFrame(integrity_rows)
display(integrity)
integrity.to_csv(AUDITED_ROOT / "paired_model_integrity.csv", index=False)


,scope,seed,n_genes,n_dev,n_cancer,hidden_size,query_domain_rule,gene_only_random_replacement
0,joint,17,15376,34760,28131,256,DEV_for_development__CANCER_for_burdziak,True
1,joint,42,15376,34760,28131,256,DEV_for_development__CANCER_for_burdziak,True
2,development_only,17,15376,34760,28131,256,DEV_for_all_queries,True
3,development_only,42,15376,34760,28131,256,DEV_for_all_queries,True


## 7. Write the downstream manifest

AE2–AE5 should use this manifest. This keeps the definitive data/model provenance in one place.


In [ ]:
manifest = {
    "pipeline_version": "audited_mlm_v1",
    "audited_root": str(AUDITED_ROOT),
    "dev_balanced_h5ad": str(DEV_BALANCED_CLEAN),
    "dev_h5ad": str(DEV_ALIGNED_CLEAN),
    "cancer_h5ad": str(CANCER_ALIGNED_CLEAN),
    "dev_baseline": str(PREPROC_OUTDIR / "dev_baseline.npy"),
    "gene_vocabulary": str(PREPROC_OUTDIR / "gene_vocabulary.npy"),
    "n_dev_cells": int(A_dev_aln.n_obs),
    "n_cancer_cells": int(A_can_aln.n_obs),
    "n_genes": int(A_dev_aln.n_vars),
    "seeds": [int(s) for s in SEEDS],
    "primary_seed": int(SEEDS[0]),
    "models": model_paths,
    "training_corrections": {
        "conjunctival_goblet_removed_before_balancing": True,
        "duplicate_ensembl_handling": "keep_first__legacy_sum_argument_removed",
        "mlm_random_replacement_gene_tokens_only": True,
    },
    "developmental_directions": {
        "defined_downstream": True,
        "not_trained_into_encoder": True,
        "specification_notebook": "AE3.ipynb",
    },
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("Wrote:", MANIFEST_PATH)
display(pd.DataFrame([
    {"scope": scope, "seed": seed, "model_dir": path}
    for scope, d in model_paths.items()
    for seed, path in d.items()
]))


Wrote: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/manifest.json


,scope,seed,model_dir
0,joint,17,/home/jfpoyatos/Work/MNCN_24_25/Development_an...
1,joint,42,/home/jfpoyatos/Work/MNCN_24_25/Development_an...
2,development_only,17,/home/jfpoyatos/Work/MNCN_24_25/Development_an...
3,development_only,42,/home/jfpoyatos/Work/MNCN_24_25/Development_an...


# AE1b closure

After this notebook completes successfully:

- the original notebooks remain the historical audit trail;
- `AE1.ipynb` explains why this retraining was performed;
- **the clean joint and development-only models are the only MLM encoders used for definitive AE2–AE6 revision analyses**;
- the actual training corrections are removal of the unintended conjunctival-goblet inclusion and restriction of random MLM replacements to gene-token IDs;
- duplicate Ensembl aliases retain the original **keep-first** handling, while the misleading legacy `duplicate_agg="sum"` interface is removed;
- AE3's biological state definitions remain fixed, but their numerical PCA coordinates, developmental vectors and malignant-alignment values are recomputed inside the clean embeddings;
- AE6 should not begin until AE2–AE5 have been rerun against this manifest.
